# KAN-REC — Comparativa completa de encoders

Entrena los tres encoders numéricos (normalización directa, AutoDis y KAN-REC)
con tres semillas sobre el mismo backbone, leyendo `criteo_10m.tsv` desde Google
Drive.

La normalización replica en pandas la de `fabric/01_spark_ingest_mlllib.py`.
Comparadas sobre los mismos datos, difieren como mucho en 1e-7.

In [ ]:
# Commit fijado: Fabric y Colab ejecutan el mismo codigo del paquete.
get_ipython().system('pip install --quiet "git+https://github.com/bdm-lab-cap/kanrec.git@main"')
get_ipython().system('pip install --quiet mlflow scikit-learn')


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import os

TSV_PATH = "/content/drive/MyDrive/kanrec/criteo_10m.tsv"
CKPT_DIR = "/content/drive/MyDrive/kanrec_checkpoints"
RESULTS_DIR = "/content/drive/MyDrive/kanrec_results"
os.makedirs(CKPT_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)

if not os.path.exists(TSV_PATH):
    raise FileNotFoundError(
        f"No encuentro {TSV_PATH}. Ajusta TSV_PATH a donde tengas "
        f"criteo_10m.tsv en Drive."
    )
size_gb = os.path.getsize(TSV_PATH) / 1e9
print(f"Fichero encontrado: {TSV_PATH} ({size_gb:.2f} GB)")


In [ ]:
import time

import mlflow
import numpy as np
import pandas as pd
import torch
from sklearn.metrics import log_loss, roc_auc_score
from torch.utils.data import DataLoader, Dataset

from kanrec.baselines import build_model

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"PyTorch: {torch.__version__} | Device: {device}")
if device.type == "cpu":
    print("Sin GPU: el entrenamiento sera muy lento.")

NUMERICAL_COLS   = [f"I{i}" for i in range(1, 14)]
CATEGORICAL_COLS = [f"C{i}" for i in range(1, 27)]
LOG_COLS = [f"I{i}" for i in range(1, 6)]   # log1p
STD_COLS = [f"I{i}" for i in range(6, 14)]  # estandarizacion
idx_cols = [f"{c}_idx" for c in CATEGORICAL_COLS]

# Filas del TSV; None usa el fichero entero. Con 1,5M los nueve modelos
# tardan ~45 min en una T4; con los 10M, ~5 h.
N_SAMPLE = 1_500_000


def replicate_fabric_01(tsv_path, num_cols, cat_cols, log_cols, std_cols,
                         seed=42, n_sample=None):
    """
    Normalizacion en pandas equivalente a la de fabric/01: imputacion a 0 y
    valor absoluto, log1p en I1..I5, estandarizacion de I6..I13 con los
    estadisticos de train, e indexado de categoricas por frecuencia
    descendente con desempate alfabetico. Las categorias no vistas y los nulos
    van a un indice reservado, como handleInvalid="keep".

    La particion train/val/test no coincide fila a fila con la de Spark
    (generadores de aleatoriedad distintos), pero la transformacion si.
    """
    cols = ["label"] + num_cols + cat_cols
    print("Leyendo el TSV...")
    t0 = time.time()
    read_kwargs = dict(sep="\t", header=None, names=cols,
                        na_values=[""], keep_default_na=True)
    if n_sample is not None:
        # Muestreo aleatorio tras leer el fichero entero: un nrows= directo
        # tomaria las primeras filas, que arrastran el orden temporal.
        df = pd.read_csv(tsv_path, **read_kwargs)
        df = df.sample(n=min(n_sample, len(df)), random_state=seed).reset_index(drop=True)
    else:
        df = pd.read_csv(tsv_path, **read_kwargs)
    print(f"  {len(df):,} filas leidas en {time.time()-t0:.0f}s")

    for c in num_cols:
        df[c] = df[c].fillna(0.0).abs()

    n = len(df)
    rng = np.random.RandomState(seed)
    perm = rng.permutation(n)
    n_train, n_val = int(n * 0.8), int(n * 0.1)
    train_df = df.iloc[perm[:n_train]].reset_index(drop=True)
    val_df   = df.iloc[perm[n_train:n_train + n_val]].reset_index(drop=True)
    test_df  = df.iloc[perm[n_train + n_val:]].reset_index(drop=True)
    del df

    for split_df in (train_df, val_df, test_df):
        for c in log_cols:
            split_df[c] = np.log1p(np.maximum(split_df[c].values, 0.0))

    means = train_df[std_cols].mean()
    stds  = train_df[std_cols].std(ddof=1)
    for split_df in (train_df, val_df, test_df):
        for c in std_cols:
            split_df[c] = (split_df[c] - means[c]) / stds[c]

    # Cardinalidad = categorias de train + 1 para el indice reservado. Con
    # train_df[c].max()+1 quedaria corta si ninguna fila de train cayera ahi,
    # y el embedding abortaria al recibirlo desde val/test.
    cardinalities = []
    for c in cat_cols:
        counts = train_df[c].value_counts()
        ordered = sorted(counts.index, key=lambda v: (-counts[v], v))
        code_map = {v: i for i, v in enumerate(ordered)}
        invalid_code = len(ordered)
        cardinalities.append(invalid_code + 1)
        for split_df in (train_df, val_df, test_df):
            split_df[f"{c}_idx"] = split_df[c].map(code_map).fillna(invalid_code).astype("int64")
        for split_df in (train_df, val_df, test_df):
            split_df.drop(columns=[c], inplace=True)

    return train_df, val_df, test_df, cardinalities


train_df, val_df, test_df, cat_cardinalities = replicate_fabric_01(
    TSV_PATH, NUMERICAL_COLS, CATEGORICAL_COLS, LOG_COLS, STD_COLS,
    seed=42, n_sample=N_SAMPLE,
)

for split_name, split_df in [("train", train_df), ("val", val_df), ("test", test_df)]:
    ctr = split_df["label"].mean()
    print(f"{split_name}: {len(split_df):,} filas | CTR={ctr:.2%}")

print("\nI6..I13 en train (esperado mean~0, std~1):")
print(train_df[STD_COLS].agg(["mean", "std"]).round(4))

print(f"\nCardinalidades categoricas: {cat_cardinalities}")
for _split_name, _split in [("val", val_df), ("test", test_df)]:
    for _j, _c in enumerate(idx_cols):
        _mx = int(_split[_c].max())
        assert _mx < cat_cardinalities[_j], (
            f"{_split_name}.{_c} tiene indice {_mx} >= cardinalidad "
            f"{cat_cardinalities[_j]}: nn.Embedding abortaria en GPU"
        )
print("Indices categoricos de val/test dentro de rango.")


In [ ]:
class CriteoDataset(Dataset):
    """DataFrame normalizado como tensores de entrada al modelo."""

    def __init__(self, df: pd.DataFrame, num_cols: list[str], idx_cols: list[str]):
        self.x_num = torch.tensor(df[num_cols].fillna(0).values.astype("float32"))
        present = [c for c in idx_cols if c in df.columns]
        self.x_cat = torch.tensor(df[present].fillna(0).values.astype("int64")) \
            if present else torch.zeros(len(df), len(idx_cols), dtype=torch.long)
        self.y = torch.tensor(df["label"].values.astype("float32"))

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return self.x_num[idx], self.x_cat[idx], self.y[idx]


BATCH_SIZE = 2048
train_ds = CriteoDataset(train_df, NUMERICAL_COLS, idx_cols)
val_ds   = CriteoDataset(val_df,   NUMERICAL_COLS, idx_cols)
test_ds  = CriteoDataset(test_df,  NUMERICAL_COLS, idx_cols)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2 if torch.cuda.is_available() else 0)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2 if torch.cuda.is_available() else 0)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, num_workers=2 if torch.cuda.is_available() else 0)


In [ ]:
# Learning rate por encoder: con 1e-3 para los tres, AutoDis queda
# infraentrenado incluso a 30 epocas.
DEFAULT_LR = {"raw": 1e-3, "autodis": 1e-2, "kan-bspline": 1e-3}


def evaluate(model, loader) -> tuple[float, float]:
    model.eval()
    preds, labels = [], []
    with torch.no_grad():
        for x_num, x_cat, y in loader:
            logits = model(x_num.to(device), x_cat.to(device)).squeeze()
            p = torch.sigmoid(logits).cpu().numpy()
            preds.extend(np.atleast_1d(p))
            labels.extend(y.numpy())
    preds = np.asarray(preds, dtype=float)
    labels = np.asarray(labels, dtype=float)
    # Las predicciones no finitas se descartan en vez de romper roc_auc_score.
    finite = np.isfinite(preds)
    if not finite.all():
        n_bad = int((~finite).sum())
        print(f"      aviso: {n_bad} predicciones no finitas descartadas en eval", flush=True)
    if finite.sum() == 0:
        return 0.5, float("inf")
    preds, labels = preds[finite], labels[finite]
    return roc_auc_score(labels, preds), log_loss(labels, preds)


def train_one_run(
    encoder_name: str,
    seed: int,
    max_epochs: int = 30,
    patience: int = 3,
    lr: float | None = None,
    embedding_dim: int = 16,
    grid_size: int = 10,
) -> dict:
    torch.manual_seed(seed)
    lr = lr if lr is not None else DEFAULT_LR[encoder_name]

    model = build_model(
        encoder=encoder_name,
        num_numerical=len(NUMERICAL_COLS),
        cat_cardinalities=cat_cardinalities,
        embedding_dim=embedding_dim,
        kan_grid_size=grid_size,
    ).to(device)

    # Calibracion del grid, solo kan-bspline. Se acumula por filas y no por
    # lotes: I6 e I12 alcanzan ~690 desviaciones tipicas.
    if hasattr(model, "calibrate"):
        CALIB_ROWS = 50_000
        calib_batches, calib_n = [], 0
        for x_num, _, _ in train_loader:
            calib_batches.append(x_num)
            calib_n += x_num.size(0)
            if calib_n >= CALIB_ROWS:
                break
        model.calibrate(torch.cat(calib_batches, dim=0).to(device))

    # El spline recibe un lr 25x mayor: arranca ~50x mas pequeno que la ruta
    # base y con un lr compartido nunca despega.
    if hasattr(model, "parameter_groups"):
        optimizer = torch.optim.Adam(model.parameter_groups(base_lr=lr), weight_decay=1e-5)
    else:
        optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-5)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=2, factor=0.5)
    # BCEWithLogitsLoss es estable: con BCELoss, una prediccion saturada a 0
    # o 1 da log(0) y aborta en GPU.
    criterion = torch.nn.BCEWithLogitsLoss()

    ckpt_path = f"{CKPT_DIR}/best_{encoder_name}_gs{grid_size}_s{seed}.pt"
    best_val_auc, patience_ctr = 0.0, 0
    t0 = time.time()

    with mlflow.start_run(run_name=f"{encoder_name}-s{seed}"):
        mlflow.log_params({
            "encoder": encoder_name, "seed": seed, "n_train": len(train_ds),
            "embedding_dim": embedding_dim, "grid_size": grid_size, "lr": lr,
            "device": str(device),
        })

        for epoch in range(max_epochs):
            model.train()
            train_loss = 0.0
            for x_num, x_cat, y in train_loader:
                x_num, x_cat, y = x_num.to(device), x_cat.to(device), y.to(device)
                optimizer.zero_grad()
                logits = model(x_num, x_cat).squeeze()
                loss = criterion(logits, y)
                if hasattr(model, "entropy_regularization_loss"):
                    loss = loss + model.entropy_regularization_loss()
                # Una perdida no finita salta el lote, no propaga NaN.
                if not torch.isfinite(loss):
                    optimizer.zero_grad()
                    continue
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                optimizer.step()
                train_loss += loss.item()

            val_auc, val_ll = evaluate(model, val_loader)
            scheduler.step(1 - val_auc)
            mlflow.log_metrics({"train_loss": train_loss, "val_auc": val_auc, "val_logloss": val_ll}, step=epoch)
            print(f"      epoch {epoch+1:2d}/{max_epochs} | val_auc={val_auc:.4f} | val_ll={val_ll:.4f}", flush=True)

            if val_auc > best_val_auc:
                best_val_auc, patience_ctr = val_auc, 0
                torch.save(model.state_dict(), ckpt_path)
            else:
                patience_ctr += 1
                if patience_ctr >= patience:
                    break

        model.load_state_dict(torch.load(ckpt_path, map_location=device))
        test_auc, test_ll = evaluate(model, test_loader)
        elapsed = time.time() - t0
        n_params = sum(p.numel() for p in model.parameters())
        mlflow.log_metrics({"test_auc": test_auc, "test_logloss": test_ll,
                             "train_seconds": elapsed, "n_params": n_params})

    print(f"  {encoder_name:12s} seed={seed:<4d} epochs={epoch+1:<3d} "
          f"val_auc={best_val_auc:.4f} test_auc={test_auc:.4f} test_ll={test_ll:.4f} "
          f"({elapsed:.0f}s)")

    return {"encoder": encoder_name, "seed": seed, "epochs": epoch + 1,
            "val_auc": best_val_auc, "test_auc": test_auc, "test_logloss": test_ll,
            "train_seconds": elapsed, "n_params": n_params, "lr": lr}


In [ ]:
# mlflow>=3.0 bloquea el backend de fichero; sqlite es la via soportada.
mlflow.set_tracking_uri(f"sqlite:///{RESULTS_DIR}/mlflow.db")
mlflow.set_experiment("kanrec-model-comparison-colab")

ENCODERS = ["raw", "autodis", "kan-bspline"]
SEEDS = [42, 123, 256]

print(f"Entrenando {len(ENCODERS)} x {len(SEEDS)} = {len(ENCODERS)*len(SEEDS)} modelos...")
print("(mismo backbone y mismos datos: la unica variable es el encoder)\n")

# Guardado incremental: al reanudar, los modelos ya completados se saltan.
import os

results = []
partial_csv = f"{RESULTS_DIR}/experiment_results_partial.csv"
if os.path.exists(partial_csv):
    results = pd.read_csv(partial_csv).to_dict("records")
    done = {(r["encoder"], r["seed"]) for r in results}
    print(f"Reanudando: {len(done)} modelo(s) ya completado(s), se saltan.\n")
else:
    done = set()

total = len(ENCODERS) * len(SEEDS)
for encoder_name in ENCODERS:
    for seed in SEEDS:
        if (encoder_name, seed) in done:
            continue
        print(f"[{len(results)+1}/{total}] {encoder_name} seed={seed} ...", flush=True)
        results.append(train_one_run(encoder_name, seed))
        pd.DataFrame(results).to_csv(partial_csv, index=False)

print(f"\n{len(results)}/{total} modelos entrenados.")


In [ ]:
results_df = pd.DataFrame(results)
results_df["timestamp"] = pd.Timestamp.now().isoformat()
results_df["source"] = "colab-pro-gpu"

summary = results_df.groupby("encoder").agg(
    test_auc_mean=("test_auc", "mean"), test_auc_std=("test_auc", "std"),
    test_logloss_mean=("test_logloss", "mean"), test_logloss_std=("test_logloss", "std"),
    train_seconds_mean=("train_seconds", "mean"),
).round(4)

print("=" * 70)
print("RESUMEN -- media +/- desviacion sobre 3 semillas (42, 123, 256)")
print("=" * 70)
print(summary.to_string())
results_df.to_csv(f"{RESULTS_DIR}/experiment_results_colab.csv", index=False)
summary.to_csv(f"{RESULTS_DIR}/experiment_summary_colab.csv")
print(f"\nGuardado en {RESULTS_DIR}/experiment_results_colab.csv")



In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

for ax, metric, title in [(axes[0], "test_auc", "AUC (test)"),
                            (axes[1], "test_logloss", "Log-loss (test)")]:
    means = results_df.groupby("encoder")[metric].mean()
    stds  = results_df.groupby("encoder")[metric].std()
    order = ["raw", "autodis", "kan-bspline"]
    ax.bar(order, means[order], yerr=stds[order], capsize=5,
           color=["#9CA3AF", "#F59E0B", "#2563EB"])
    ax.set_title(title)
    ax.set_ylabel(metric)

plt.tight_layout()
plt.savefig(f"{RESULTS_DIR}/comparison_figure.png", dpi=150)
plt.show()
print(f"Figura guardada en {RESULTS_DIR}/comparison_figure.png")


In [ ]:
# Ablacion de grid_size sobre KAN-REC.
ablation_results = []
for grid_size in [5, 10, 20]:
    r = train_one_run("kan-bspline", seed=42, grid_size=grid_size, max_epochs=20, patience=2)
    r["grid_size"] = grid_size
    ablation_results.append(r)

ablation_df = pd.DataFrame(ablation_results)
print("\nAblacion de grid_size (KAN-REC, seed=42):")
print(ablation_df[["grid_size", "test_auc", "test_logloss", "train_seconds"]].to_string(index=False))

ablation_df.to_csv(f"{RESULTS_DIR}/gridsize_ablation_colab.csv", index=False)
print(f"\nGuardado en {RESULTS_DIR}/gridsize_ablation_colab.csv")


## Salidas

- `experiment_results_colab.csv` y `experiment_summary_colab.csv`: resultados por
  ejecución y agregados por encoder.
- `comparison_figure.png`: figura de la comparativa.
- `gridsize_ablation_colab.csv`: ablación de `grid_size` sobre KAN-REC.

Los resultados de Fabric se obtienen con menos filas, menos épocas y una sola
semilla: son la prueba del pipeline, no la comparativa.